# Importing Libraries

In [16]:
# Importing Libraries

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

# Importing Dataset

In [17]:
# Importing Dataset

df = pd.read_csv('heart.csv')

df.sample(10)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
385,61,M,ASY,150,0,0,Normal,105,Y,0.0,Flat,1
420,66,M,NAP,110,213,1,LVH,99,Y,1.3,Flat,0
398,52,M,ASY,165,0,1,Normal,122,Y,1.0,Up,1
791,51,M,ASY,140,298,0,Normal,122,Y,4.2,Flat,1
292,53,M,ASY,130,182,0,Normal,148,N,0.0,Up,0
497,61,M,ASY,146,241,0,Normal,148,Y,3.0,Down,1
36,65,M,ASY,140,306,1,Normal,87,Y,1.5,Flat,1
131,46,M,ASY,110,202,0,Normal,150,Y,0.0,Flat,1
880,52,M,NAP,172,199,1,Normal,162,N,0.5,Up,0
260,46,M,ATA,140,275,0,Normal,165,Y,0.0,Up,0


# Basic Exploration

In [18]:
# Basic Exploration

print("Information : \n")
print(df.info())

print("\nMissing Values : \n")
print(df.isnull().sum())

print("\nDuplicate Entries : \n")
print(df.duplicated().sum())

Information : 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB
None

Missing Values : 

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
E

# Data Preprocessing

### Splitting the Data

In [19]:
# Splitting the data

x = df.drop('HeartDisease', axis = 1)
y = df['HeartDisease']

x_train,x_test,y_train,y_test = train_test_split(x,y,
                                                test_size = 0.2,
                                                stratify = y,
                                                random_state = 20)

### Transforming the Data

In [20]:
# Transforming the data

one_hot_columns = ['Sex','ChestPainType','RestingECG','ExerciseAngina','ST_Slope']
standard_columns = ['Age','RestingBP','Cholesterol','MaxHR','Oldpeak']

trf = ColumnTransformer(transformers = [
    ('standard',StandardScaler(),standard_columns),
    ('onehot',OneHotEncoder(handle_unknown = 'ignore', drop = 'if_binary'),one_hot_columns)
],remainder = 'passthrough')

# Model - VotingClassifier

In [21]:
# Importing the Models

clf1 = LogisticRegression()
clf2 = DecisionTreeClassifier()
clf3 = GaussianNB()

In [22]:
model = VotingClassifier(estimators = [
    ('LR',clf1),
    ('DTC',clf2),
    ('GNB',clf3)
],voting = 'soft')

In [23]:
# Creating Pipeline

pipe = Pipeline([
    ('transform',trf),
    ('model',model)
])

In [24]:
# Training the Model

pipe.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('transform', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('standard', ...), ('onehot', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [25]:
# Model Prediction
predictions = pipe.predict(x_test)

In [26]:
# Model Evaluation

print("Accuracy Score : ",accuracy_score(y_test,predictions))
print("\nConfusion Matrix : \n",confusion_matrix(y_test,predictions))
print("\nF1_score : \n", f1_score(y_test,predictions))

Accuracy Score :  0.8804347826086957

Confusion Matrix : 
 [[70 12]
 [10 92]]

F1_score : 
 0.8932038834951457
